# 00 · Setup y base de datos
## Módulo 3 — Programa Data Science for Business | FUTURA

---

Este notebook es el **punto de entrada** de toda la cadena. No hace análisis: prepara el
terreno para que las 9 etapas siguientes trabajen sobre una base **limpia, tipada y
reproducible**.

### La cadena completa

| NB | Etapa | Entrada | Salida |
|----|-------|---------|--------|
| **00** | **Setup** | `loan_prediction.csv` | `00_base` |
| 01 | Caso de uso + muestreo | `00_base` | `01_train`, `01_test`, modelo de segmento |
| 02 | EDA univariado + bivariado | `01_*` | decisiones de EDA |
| 03 | Limpieza dura (por segmento) | `01_*` | `03_train`, `03_test` |
| 04 | Outliers multivariados | `03_*` | `04_train`, `04_test`, detectores |
| 05 | WoE e IV | `04_*` | features WoE, tabla IV |
| 06 | Pipeline de preprocesamiento | `04_*` | `06_train_proc`, `06_test_proc` |
| 07 | Selección de variables | `06_*` + IV | variables seleccionadas |
| 08 | PCA + MCA | `06_*`, `03_*` | coordenadas, contribuciones, cos2 |
| 09 | Balanceo | `07_*` | train balanceados |
| 10 | Consolidación | todo | dataset final + linaje |

### El contrato de traspaso

Cada etapa se comunica con la siguiente por **tres canales**, centralizados en `pipeline_io.py`:

- **`data/*.parquet`** → los DataFrames. Usamos **parquet**, no CSV: preserva los `dtypes` y el
  índice. Un CSV vuelve todo a texto/float y resetea el índice — fuente clásica de bugs de
  desalineación.
- **`artifacts/*.joblib`** → los **modelos y parámetros aprendidos** (segmentador, imputers,
  scalers, WoE, PCA, MCA). Esto es lo que se despliega a producción.
- **`reports/*.json`** → las **decisiones** que una etapa toma y la siguiente necesita
  (columnas descartadas, ranking de IV, variables seleccionadas…).

In [14]:
#!pip install -r requirements.txt

## 0.1 Verificación del entorno

In [15]:
import sys
import warnings
warnings.filterwarnings("ignore")

from pipeline_io import (
    SEED, RAW_CSV, ID_COL, TARGET, DATA, ARTIFACTS, REPORTS,
    save_stage, set_plot_theme,
)

import pandas as pd
import numpy as np

set_plot_theme()
np.random.seed(SEED)

print("Python     :", sys.version.split()[0])
print("pandas     :", pd.__version__)
print("numpy      :", np.__version__)
print("Semilla    :", SEED)
print("Dataset    :", RAW_CSV.name, "->", "EXISTE" if RAW_CSV.exists() else "NO ENCONTRADO")
print("Carpetas   : data/, artifacts/, reports/ listas")

assert RAW_CSV.exists(), (
    f"No se encontró 'loan_prediction.csv'. Debe estar en la misma carpeta que estos "
    f"notebooks (junto a pipeline_io.py). Ruta buscada: {RAW_CSV}"
)

Python     : 3.13.12
pandas     : 3.0.3
numpy      : 2.4.6
Semilla    : 2021
Dataset    : loan_prediction.csv -> EXISTE
Carpetas   : data/, artifacts/, reports/ listas


Verificamos que todas las librerías especializadas que usará la cadena están disponibles.
Mejor fallar aquí, en el setup, que en la celda 40 del notebook 8.

In [16]:
import importlib

# Librerías que usa la cadena. 'prince' es para el MCA (NB08); 'missingno' para el mapa de
# nulos (NB02). El resto viene con cualquier instalación estándar de ciencia de datos.
requeridas = ["pandas", "numpy", "sklearn", "scipy", "statsmodels",
              "imblearn", "seaborn", "matplotlib", "prince", "missingno"]
faltan = []
for lib in requeridas:
    try:
        m = importlib.import_module(lib)
        print(f"  ✓ {lib:12s} {getattr(m, '__version__', '?')}")
    except ImportError:
        faltan.append(lib)
        print(f"  ✗ {lib:12s} NO INSTALADO")
if faltan:
    print(f"\n⚠️  Faltan librerías: {faltan}")
    print(f"   Instálalas con:  pip install {' '.join(faltan)}")

assert not faltan, f"Faltan librerías: {faltan}. Instálalas y reinicia el kernel."
print("\nEntorno completo. ✓")

  ✓ pandas       3.0.3
  ✓ numpy        2.4.6
  ✓ sklearn      1.8.0
  ✓ scipy        1.17.1
  ✓ statsmodels  0.14.6
  ✓ imblearn     0.14.2
  ✓ seaborn      0.13.2
  ✓ matplotlib   3.10.9
  ✓ prince       0.20.1
  ✓ missingno    0.5.2

Entorno completo. ✓


## 0.2 Lectura del dataset crudo

In [17]:
df = pd.read_csv(RAW_CSV)
print("Shape crudo:", df.shape)
df.head()

Shape crudo: (614, 13)


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    str    
 1   Gender             601 non-null    str    
 2   Married            611 non-null    str    
 3   Dependents         599 non-null    str    
 4   Education          614 non-null    str    
 5   Self_Employed      582 non-null    str    
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    str    
 12  Loan_Status        614 non-null    str    
dtypes: float64(4), int64(1), str(8)
memory usage: 83.4 KB


## 0.3 Sanity checks

Antes de tocar nada, tres preguntas que evitan sorpresas después:

In [19]:
print("¿El ID es único por fila?     ",
      df[ID_COL].is_unique, f"({df[ID_COL].nunique()} de {len(df)})")
print("¿Hay filas completamente duplicadas?",
      df.drop(columns=[ID_COL]).duplicated().sum(), "duplicados")
print("¿Hay filas 100% nulas?        ",
      (df.isna().all(axis=1)).sum())
print()
print("Nulos por columna:")
print(df.isna().sum()[df.isna().sum() > 0].to_string())

¿El ID es único por fila?      True (614 de 614)
¿Hay filas completamente duplicadas? 0 duplicados
¿Hay filas 100% nulas?         0

Nulos por columna:
Gender              13
Married              3
Dependents          15
Self_Employed       32
LoanAmount          22
Loan_Amount_Term    14
Credit_History      50


## 0.4 Definición y codificación del target

La variable objetivo es `Loan_Status` (Y/N). La codificamos como numérica con una convención
**deliberada**:

- `Y` = *Yes*, crédito **aprobado** → **0**
- `N` = *No*, crédito **rechazado** → **1**

> **¿Por qué el rechazo es el `1`?** Por convención de riesgo, el `1` se reserva para el
> **evento de interés** (el caso "malo" que queremos detectar). Así la salida del modelo se lee
> directo como *"probabilidad de que la solicitud sea rechazada"*, y las métricas WoE / IV / KS
> asumen esa orientación. Si se invierte el mapeo, se invierten todos los signos.
> **Se decide una vez, se documenta, y no se toca.**

In [20]:
print("Antes:", df[TARGET].value_counts().to_dict())
df[TARGET] = df[TARGET].map({"Y": 0, "N": 1}).astype("int8")
print("Después:", df[TARGET].value_counts().to_dict())
print(f"\nTasa de evento (rechazo): {df[TARGET].mean():.2%}")

Antes: {'Y': 422, 'N': 192}
Después: {0: 422, 1: 192}

Tasa de evento (rechazo): 31.27%


## 0.5 Tipado explícito

`pandas` infiere tipos, pero conviene ser explícito. En particular, `Credit_History` viene como
`0.0/1.0` y `pandas` la lee como numérica — pero es **binaria/categórica**. La dejamos como
`float` por ahora (tiene nulos) y decidiremos su tratamiento en el notebook 06; lo importante es
**saber que no es una variable continua** aunque el `dtype` diga lo contrario.

In [21]:
categoricas = ["Gender", "Married", "Dependents", "Education", "Self_Employed", "Property_Area"]
for c in categoricas:
    df[c] = df[c].astype("string")

print("Tipos finales:")
print(df.dtypes.to_string())

Tipos finales:
Loan_ID                  str
Gender                string
Married               string
Dependents            string
Education             string
Self_Employed         string
ApplicantIncome        int64
CoapplicantIncome    float64
LoanAmount           float64
Loan_Amount_Term     float64
Credit_History       float64
Property_Area         string
Loan_Status             int8


## 0.6 Exportación de la base

`00_base.parquet` es el punto de partida de toda la cadena: dataset completo, target codificado,
tipos correctos, `Loan_ID` **conservado** (lo usaremos como índice para poder rastrear cada
cliente a lo largo de las etapas).

In [22]:
df = df.set_index(ID_COL)
ruta = save_stage(df, "00_base")

print("Base exportada:")
print(f"  {ruta}")
print(f"  {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"  índice = {df.index.name} (rastreable en toda la cadena)")
print()
print("Siguiente paso → 01_caso_muestreo.ipynb")

Base exportada:
  C:\Users\carlo\OneDrive\Futura\claude-futura\temp_clase_2_modulo_3\output\data\00_base.csv
  614 filas × 12 columnas
  índice = Loan_ID (rastreable en toda la cadena)

Siguiente paso → 01_caso_muestreo.ipynb
